# Colab runner: smoke test, pilot and Phases A to D

Before running: **Runtime > Change runtime type > GPU** (L4 or A100 if you have them; T4 works),
and put your Kaggle token as `aml-gnn-transfer/kaggle.json` in your Google Drive.
Colab does not run in the background: keep the tab open. If it disconnects, run all cells again:
finished runs have records on Drive and are skipped, an interrupted run's files are moved aside
(never deleted), and the data, prepared graphs and models are reused from Drive.

Every step stops the notebook if it fails. This file is written by `python -m tools.make_notebooks`.

Pick `RUN_SET` in the first cell:

| RUN_SET | What it runs |
| --- | --- |
| `smoke` (default) | source, then 10% scratch, then 10% fine-tune, each capped at 2 epochs; checks the pair used the same labels and steps |
| `pilot` | GIN pilot: one source model, 10/25/100% labels, scratch vs fine-tune |
| `phaseB` | zero-shot: the pilot's GIN source model scored on LI-Small test, no training |
| `phaseA` | GIN at 100% labels, 20-epoch budget, scratch and fine-tune (ids end `_b20`) |
| `phaseC` | GIN grid: `PHASE_C_FRACTIONS` x 2 setups x `PHASE_C_SEEDS` (10/25/100%, seeds 0 and 1); needs `BUDGET_C` in `src/config.py`; reuses pilot (budget 8) or Phase A (budget 20) runs |
| `phaseD` | PNA: trains its HI-Small source model, then the same grid; needs `BUDGET_C` |
| `export_curves` | no GPU: rebuilds validation curves for the pilot, Phase A and Phase C from their Multi-GNN logs on Drive, into `<set>/curves/` for `pilot`, `phaseA` and `phaseC` |

Each run prints an estimate before it starts, then its real minutes and Colab units with a running total.
A run expected to pass 3 GPU hours stops unless `ALLOW_LONG_RUNS = True` (AGENTS.md rule 6: ask the owner first).
Phases A to D start from the pilot's source model on Drive, so run the pilot's source run first.

In [ ]:
import os, shlex, subprocess, sys, time


def run(command):
    """Run a shell command, show its output as it arrives, and stop the notebook if it fails."""
    process = subprocess.Popen(
        ["bash", "-o", "pipefail", "-c", command],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
    )
    for line in process.stdout:
        print(line, end="", flush=True)
    code = process.wait()
    if code != 0:
        raise RuntimeError(f"Command failed (exit code {code}): {command}")


REPO_URL = "https://github.com/coachuttam/aml-gnn-transfer.git"

RUN_SET = "smoke"  # "smoke" | "pilot" | "phaseA" | "phaseB" | "phaseC" | "phaseD" | "export_curves"
D_SEEDS = None  # phaseD only, e.g. [0, 1, 2]; None = PHASE_D_SEEDS in src/config.py
ALLOW_LONG_RUNS = False  # True only after the owner approved a run over 3 GPU hours
assert RUN_SET in ("smoke", "pilot", "phaseA", "phaseB", "phaseC", "phaseD", "export_curves"), RUN_SET
NEEDS_GPU = RUN_SET != "export_curves"  # export_curves only reads logs on Drive
WORK = "/content"
REPO = f"{WORK}/repo"

from google.colab import drive
drive.mount("/content/drive")
DRIVE = "/content/drive/MyDrive/aml-gnn-transfer"  # kaggle.json lives here; everything we keep is saved here
os.makedirs(DRIVE, exist_ok=True)

# Data, prepared graphs, models and records all live on Drive, so a disconnect loses nothing.
# Each run set keeps its own records and models (configs/phases.yaml); src.run_phase sets those folders.
os.environ["AML_DATA_ROOT"] = f"{DRIVE}/data"
os.environ["AML_GRAPH_CACHE"] = f"{DRIVE}/graph_cache"
os.environ["PYTHONDONTWRITEBYTECODE"] = "1"
if RUN_SET == "smoke":  # smoke runs go to a throw-away folder
    SMOKE_HOME = f"{DRIVE}/smoke/{time.strftime('%Y%m%d-%H%M%S')}"
    os.environ["AML_MODEL_ROOT"] = f"{SMOKE_HOME}/models"
    os.environ["AML_RUN_RECORDS_DIR"] = f"{SMOKE_HOME}/records"
print("Run set:", RUN_SET)
if NEEDS_GPU:
    # Stop here if the GPU accelerator is off; remember which GPU we got.
    run("nvidia-smi --query-gpu=name,driver_version,memory.total --format=csv")
    GPU_NAME = subprocess.check_output(
        ["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"], text=True
    ).splitlines()[0].strip()
    print("GPU:", GPU_NAME)
else:
    GPU_NAME = "none"; print("export_curves: no GPU needed")

In [ ]:
# Always run the latest pushed code: update an existing checkout instead of keeping a stale one.
if os.path.isdir(REPO):
    run(f"git -C {REPO} fetch origin && git -C {REPO} reset --hard origin/main")
else:
    run(f"git clone {REPO_URL} {REPO}")
run(f"git -C {REPO} submodule update --init")
run(f"git -C {REPO} log -1 --oneline")
run(f"git -C {REPO} submodule status")
os.chdir(REPO)
sys.path.insert(0, REPO)

Environment. First choice is Multi-GNN's own `env.yml` (micromamba, forced to a CUDA build of torch),
plus pyg-lib (the fast neighbour sampler, which `env.yml` lacks) as a wheel matched to that torch.
Whatever we get must pass `src.check_environment`: CUDA works, the PyG extensions import and run,
pyg-lib is in use. If it does not, we fall back to the notebook's own CUDA torch with all PyG add-ons
installed as matching wheels (`src.pyg_wheels`). If that also fails the check, the notebook stops.

In [ ]:
if NEEDS_GPU:
    MICROMAMBA = f"{WORK}/bin/micromamba"
    ENV_PYTHON = f"{MICROMAMBA} run -r {WORK}/mamba -n multignn python"

    try:
        run(f"curl -Ls https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xj -C {WORK} bin/micromamba")
        run(f"{MICROMAMBA} create -y -r {WORK}/mamba -n multignn -f external/Multi-GNN/env.yml 'pytorch=*=*cuda11.8*'")
        run(f"{ENV_PYTHON} -m src.pyg_wheels --only pyg_lib")
        run(f"{ENV_PYTHON} -m src.check_environment")
        PYTHON, ENV_LABEL = ENV_PYTHON, "env.yml"
    except RuntimeError as problem:
        print("\nThe env.yml environment is not usable:", problem)
        print("Falling back to this notebook's torch with matching PyG wheels.\n")
        run(f"{sys.executable} -m src.pyg_wheels")
        run(f"{sys.executable} -m src.check_environment")  # raises if still wrong
        PYTHON, ENV_LABEL = sys.executable, "pip-fallback"
    ENVIRONMENT_LABEL = f"{ENV_LABEL} / {GPU_NAME}"
    print("Using:", ENVIRONMENT_LABEL)
else:
    PYTHON, ENVIRONMENT_LABEL = sys.executable, "none"

Download HI-Small and LI-Small from Kaggle (skipped if already on Drive) and format them with IBM's script (CPU only, slow the first time).

In [ ]:
if NEEDS_GPU:
    # The Kaggle token is read from Drive and copied where the Kaggle CLI looks. It is never printed.
    TOKEN = f"{DRIVE}/kaggle.json"
    if not os.path.exists(TOKEN):
        raise FileNotFoundError(f"Put your Kaggle API token (kaggle.json) in {DRIVE}/ first")
    os.makedirs("/root/.kaggle", exist_ok=True)
    run(f"cp {TOKEN} /root/.kaggle/kaggle.json && chmod 600 /root/.kaggle/kaggle.json")
    run(f"{sys.executable} -m pip install -q kaggle")
    run(f"{sys.executable} -m src.download_data")  # skips files already on Drive
    run(f"{PYTHON} -m src.prepare --run-id prepare_only")  # IBM's formatter, skipped if already formatted
else:
    print('export_curves: no download')

Smoke test (only when `RUN_SET = "smoke"`). The first run of each dataset builds its prepared graph (several minutes of CPU, GPU idle) and saves it to Drive.

In [ ]:
def run_timed(label, command):
    """Run one step and print how long it took and on which GPU."""
    start = time.time()
    run(command)
    print(f"[{label}] {(time.time() - start) / 60:.1f} min on {GPU_NAME}", flush=True)


def smoke_chain(python, environment_label, seed, budget, fraction):
    """Source, then scratch and finetune at `fraction`, each as a smoke run."""
    label = shlex.quote(environment_label)
    for mode in ("source", "scratch", "finetune"):
        share = 1.0 if mode == "source" else fraction
        print(f"\n===== wrapper smoke run: {mode}, fraction {share} =====")
        run_timed(
            f"smoke {mode}",
            f"{python} -m src.run_experiment --model gin --mode {mode} --fraction {share} "
            f"--seed {seed} --step-budget-epochs {budget} --environment {label} --smoke-test",
        )

if RUN_SET != "smoke":
    print(f"RUN_SET is {RUN_SET}: skipping the smoke runs")
else:
    smoke_chain(PYTHON, ENVIRONMENT_LABEL, 0, 1, 0.10)
    run(f"{sys.executable} -m src.check_smoke")

The chosen run set (anything but `smoke`). Prints the estimate first, then one estimate and one timed line per run, with units.

In [ ]:
if RUN_SET == "smoke":
    print("RUN_SET is smoke: no phase runs")
elif RUN_SET == "export_curves":
    run(f"{sys.executable} -m src.export_curves --home {DRIVE} --sets pilot phaseA phaseC")
    run(f"ls {DRIVE}/pilot/curves {DRIVE}/phaseA/curves {DRIVE}/phaseC/curves")
else:
    seeds = " ".join(str(seed) for seed in D_SEEDS) if (RUN_SET == "phaseD" and D_SEEDS) else ""
    words = [
        f"--set {RUN_SET}", f"--home {DRIVE}", f"--python {shlex.quote(PYTHON)}",
        f"--gpu-name {shlex.quote(GPU_NAME)}", f"--environment {shlex.quote(ENVIRONMENT_LABEL)}",
    ]
    if seeds:
        words.append(f"--seeds {seeds}")
    if ALLOW_LONG_RUNS:
        words.append("--allow-long")
    run(f"{sys.executable} -m src.run_phase " + " ".join(words))

What the records say. Copy that records folder into the repo (`results/pilot/records/` for the pilot, `results/phases/<set>/records/` for a phase) and run `python -m src.collect`.

In [ ]:
if RUN_SET not in ("smoke", "export_curves"):
    from pathlib import Path
    from src.phases import phase_folders
    records_dir = phase_folders(RUN_SET, Path(DRIVE)).records
    print("Records are in:", records_dir)
    run(f"grep -H -E '\"(status|minority_f1|epochs_requested|gradient_steps|kept_labels_fingerprint)\"' {records_dir}/*.json || true")